# TNBike Sales by Territory — Exploratory Data Analysis

**Adapted from WQU ADSL Module 1 (Housing in Mexico)**

In this notebook we explore TNBike sales data grouped by territory/region, analogous to exploring housing prices across Mexican states. We will:
- Connect to `tnbike_db` PostgreSQL database
- Query and inspect `fact_sales`, `dim_territory`, `dim_product`
- Visualize sales distributions by region
- Identify outliers and data quality issues

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid')

print('Libraries imported successfully.')

## 2. Load Configuration

In [ ]:
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

db = config['database']
print('Config loaded:')
print(f"  Host   : {db['host']}:{db['port']}")
print(f"  DB     : {db['dbname']}")
print(f"  Schema : {db['schema']}")

## 3. Connect to PostgreSQL Database

In [ ]:
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql://{db['user']}:{db['password']}@{db['host']}:{db['port']}/{db['dbname']}"
)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT version()"))
    print('Connected to:', result.fetchone()[0])

## 4. Query Sales Data

In [ ]:
query = """
SELECT
    fs.order_id,
    fs.order_date,
    fs.quantity,
    fs.unit_price,
    fs.total_amount,
    fs.discount,
    dt.territory_name,
    dt.region,
    dt.country,
    dp.product_name,
    dp.category,
    dp.subcategory
FROM tnbike.fact_sales fs
LEFT JOIN tnbike.dim_territory dt ON fs.territory_id = dt.territory_id
LEFT JOIN tnbike.dim_product dp   ON fs.product_id   = dp.product_id
WHERE fs.order_date BETWEEN '2025-01-01' AND '2026-03-31'
ORDER BY fs.order_date
"""

df = pd.read_sql_query(query, engine)
print(f'Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')
df.head()

## 5. Initial Data Inspection

In [ ]:
print('=== DataFrame Shape ===')
print(df.shape)
print()
print('=== Column dtypes ===')
print(df.dtypes)
print()
print('=== Missing values ===')
print(df.isnull().sum())

In [ ]:
print('=== Descriptive Statistics ===')
df[['quantity','unit_price','total_amount','discount']].describe()

## 6. Sales Distribution by Region

In [ ]:
region_sales = (
    df.groupby('region')['total_amount']
    .agg(['sum', 'mean', 'count'])
    .rename(columns={'sum': 'total_revenue', 'mean': 'avg_order', 'count': 'order_count'})
    .sort_values('total_revenue', ascending=False)
    .reset_index()
)

print('Sales by Region:')
print(region_sales.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total Revenue by Region
axes[0].bar(region_sales['region'], region_sales['total_revenue'], color='steelblue', edgecolor='white')
axes[0].set_title('Total Revenue by Region', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Total Revenue ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=30)

# Average Order Value by Region
axes[1].bar(region_sales['region'], region_sales['avg_order'], color='coral', edgecolor='white')
axes[1].set_title('Avg Order Value by Region', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Average Order ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('region_sales.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 7. Sales Distribution by Territory

In [ ]:
territory_sales = (
    df.groupby('territory_name')['total_amount']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    territory_sales,
    x='territory_name', y='total_amount',
    title='Top 10 Territories by Total Revenue',
    labels={'territory_name': 'Territory', 'total_amount': 'Total Revenue ($)'},
    color='total_amount',
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

## 8. Product Category Analysis

In [ ]:
cat_sales = (
    df.groupby('category')['total_amount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=cat_sales, x='category', y='total_amount', palette='viridis', ax=ax)
ax.set_title('Total Revenue by Product Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Total Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## 9. Distribution of total_amount (target variable)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(df['total_amount'], bins=50, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Distribution of total_amount', fontsize=12)
axes[0].set_xlabel('total_amount ($)')

sns.histplot(np.log1p(df['total_amount']), bins=50, kde=True, color='darkorange', ax=axes[1])
axes[1].set_title('Distribution of log(total_amount + 1)', fontsize=12)
axes[1].set_xlabel('log(total_amount + 1)')

plt.tight_layout()
plt.show()

print(f"Skewness (original) : {df['total_amount'].skew():.4f}")
print(f"Skewness (log)      : {np.log1p(df['total_amount']).skew():.4f}")

## 10. Correlation Heatmap

In [ ]:
numeric_cols = ['quantity', 'unit_price', 'total_amount', 'discount']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Monthly Sales Trend

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
monthly = df.resample('M', on='order_date')['total_amount'].sum().reset_index()
monthly.columns = ['month', 'revenue']

fig = px.line(
    monthly, x='month', y='revenue',
    title='Monthly Total Revenue Trend',
    labels={'month': 'Month', 'revenue': 'Revenue ($)'},
    markers=True
)
fig.update_traces(line_color='steelblue', marker_size=6)
fig.show()

## 12. Boxplot: total_amount by Region

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
order = df.groupby('region')['total_amount'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='region', y='total_amount', order=order, palette='Set2', ax=ax)
ax.set_title('Order Value Distribution by Region', fontsize=13, fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('total_amount ($)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 13. Scatter: quantity vs total_amount

In [ ]:
sample = df.sample(min(2000, len(df)), random_state=42)

fig = px.scatter(
    sample, x='quantity', y='total_amount',
    color='region',
    title='Quantity vs Total Amount (sample 2000)',
    labels={'quantity': 'Quantity', 'total_amount': 'Total Amount ($)'},
    opacity=0.6,
    trendline='ols'
)
fig.show()

## 14. Discount Analysis

In [ ]:
discount_bins = pd.cut(df['discount'], bins=[-0.001, 0, 0.05, 0.1, 0.2, 0.5, 1.0],
                       labels=['0%','0-5%','5-10%','10-20%','20-50%','50%+'])
discount_effect = df.groupby(discount_bins, observed=True)['total_amount'].mean().reset_index()
discount_effect.columns = ['discount_range', 'avg_total']

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=discount_effect, x='discount_range', y='avg_total', palette='magma', ax=ax)
ax.set_title('Average Order Value by Discount Range', fontsize=12)
ax.set_xlabel('Discount Range')
ax.set_ylabel('Avg total_amount ($)')
plt.tight_layout()
plt.show()

## 15. Key Findings Summary

In [ ]:
print('=' * 55)
print('  KEY FINDINGS — TNBike Sales EDA')
print('=' * 55)
print(f"Total orders analyzed   : {len(df):,}")
print(f"Date range              : {df['order_date'].min().date()} – {df['order_date'].max().date()}")
print(f"Total revenue           : ${df['total_amount'].sum():,.2f}")
print(f"Avg order value         : ${df['total_amount'].mean():,.2f}")
print(f"Median order value      : ${df['total_amount'].median():,.2f}")
print(f"Max order value         : ${df['total_amount'].max():,.2f}")
print(f"Number of regions       : {df['region'].nunique()}")
print(f"Number of territories   : {df['territory_name'].nunique()}")
print(f"Number of categories    : {df['category'].nunique()}")
print(f"Missing values total    : {df.isnull().sum().sum()}")
print('=' * 55)
print()
print('Top region by revenue:')
print(region_sales.iloc[0].to_dict())
print()
print('Proceed to 02_cleaning.ipynb for data preparation.')